# Hotel arrivals model training

This notebook is the training and evaluation entry point for the first hotel model. It compares conventional regressors on chronological training and validation periods, evaluates the selected model once on the later test period, then saves a model artifact and metadata. The API and Streamlit dashboard load those saved files; they do not train a model.

Target: daily completed reservation arrivals per hotel. The source does not contain room capacity, inventory snapshots, or forecast-time prices, so this is not a price elasticity or revenue model.

## 1. Paths and imports

Update DATA_PATH if the source CSV is stored somewhere else. Raw data stays in its original location. Generated model files go under the repository artifacts folder, which Git ignores.

In [ ]:
import hashlib
import json
import os
from datetime import UTC, datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from dynamic_pricing.models.hotel_arrivals import build_arrival_features, build_daily_hotel_arrivals

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
DATA_PATH = Path(r"C:\Users\aadit\OneDrive\Desktop\hotel_bookings.csv")
ARTIFACT_DIR = REPO_ROOT / "artifacts" / "models"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Set DATA_PATH to the supplied hotel_bookings.csv: {DATA_PATH}")

print(f"Reading source data from: {DATA_PATH}")
print(f"Notebook outputs will be saved to: {ARTIFACT_DIR}")

## 2. Build the daily target

Exact duplicate source rows are removed. Canceled and no-show bookings are excluded because they were not completed arrivals. The output is a count of reservation records, not a count of rooms or room-nights.

In [ ]:
bookings = pd.read_csv(DATA_PATH)
daily, data_audit = build_daily_hotel_arrivals(bookings)

print("Source and target audit:")
display(pd.Series(data_audit, name="count").to_frame())
display(daily.head())
print("Date range:", daily["arrival_at"].min().date(), "to", daily["arrival_at"].max().date())
print("Hotels:", sorted(daily["hotel"].unique()))

## 3. Leakage-aware features and chronological partitions

Every demand lag and rolling average uses only earlier calendar dates. Train, validation, and test dates are non-overlapping and shared across hotels. Validation selects the algorithm; the final test period is held aside until that choice is made.

In [ ]:
FEATURES = [
    "hotel",
    "day_of_week",
    "month",
    "is_weekend",
    "year_sin",
    "year_cos",
    "demand_lag_1",
    "demand_lag_7",
    "demand_lag_14",
    "rolling_mean_7",
    "rolling_mean_28",
]
featured = build_arrival_features(daily)
unique_dates = np.array(sorted(featured["arrival_at"].unique()))
if len(unique_dates) < 60:
    raise ValueError("At least 60 distinct arrival dates are required.")

train_end = int(len(unique_dates) * 0.70)
validation_end = int(len(unique_dates) * 0.85)
train_dates = unique_dates[:train_end]
validation_dates = unique_dates[train_end:validation_end]
test_dates = unique_dates[validation_end:]

train = featured.loc[featured["arrival_at"].isin(train_dates)].copy()
validation = featured.loc[featured["arrival_at"].isin(validation_dates)].copy()
test = featured.loc[featured["arrival_at"].isin(test_dates)].copy()

print(
    {
        "train": (str(train_dates[0])[:10], str(train_dates[-1])[:10], len(train)),
        "validation": (
            str(validation_dates[0])[:10],
            str(validation_dates[-1])[:10],
            len(validation),
        ),
        "test": (str(test_dates[0])[:10], str(test_dates[-1])[:10], len(test)),
    }
)

## 4. Train traditional regression candidates

A median predictor is retained as a simple reference. Linear Regression, Random Forest, and HistGradientBoosting are fit on the training period and compared by validation MAE. Missing early lag values are imputed inside each scikit-learn pipeline.

In [ ]:
NUMERIC_FEATURES = [name for name in FEATURES if name != "hotel"]


def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("numeric", SimpleImputer(strategy="median", add_indicator=True), NUMERIC_FEATURES),
            ("hotel", OneHotEncoder(handle_unknown="ignore"), ["hotel"]),
        ],
        remainder="drop",
    )


def make_pipeline(regressor):
    return Pipeline(
        [
            ("preprocess", make_preprocessor()),
            ("regressor", regressor),
        ]
    )


candidate_factories = {
    "linear_regression": lambda: LinearRegression(),
    "random_forest": lambda: RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=3,
        max_features=0.8,
        n_jobs=1,
        random_state=42,
    ),
    "hist_gradient_boosting": lambda: HistGradientBoostingRegressor(
        max_iter=150,
        learning_rate=0.06,
        l2_regularization=1.0,
        random_state=42,
    ),
    "dummy_median": lambda: DummyRegressor(strategy="median"),
}
try:
    from xgboost import XGBRegressor

    candidate_factories["xgboost"] = lambda: XGBRegressor(
        n_estimators=150,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        n_jobs=1,
        random_state=42,
        verbosity=0,
    )
except ImportError:
    print("XGBoost is not installed; evaluating the scikit-learn candidates only.")

X_train = train[FEATURES].copy()
X_validation = validation[FEATURES].copy()
for frame in (X_train, X_validation):
    frame["hotel"] = frame["hotel"].astype(str)

y_train = train["arrivals"].astype(float)
y_validation = validation["arrivals"].astype(float)
validation_results = {}
fitted_candidates = {}

for name, factory in candidate_factories.items():
    candidate = make_pipeline(factory())
    candidate.fit(X_train, y_train)
    predictions = np.maximum(candidate.predict(X_validation), 0.0)
    validation_results[name] = {
        "mae": float(mean_absolute_error(y_validation, predictions)),
        "rmse": float(np.sqrt(mean_squared_error(y_validation, predictions))),
    }
    fitted_candidates[name] = candidate

validation_table = pd.DataFrame(validation_results).T.sort_values("mae")
display(validation_table)
selected_model_name = validation_table.index[0]
print("Selected using validation MAE:", selected_model_name)

## 5. Evaluate once on the later test period

The seasonal naive baseline predicts the same weekday from seven days earlier, falling back to the prior day if that value is unavailable. Test features use the actual preceding daily counts, corresponding to a rolling one-day-ahead forecast that is refreshed after each day.

In [ ]:
def seasonal_naive(frame):
    lag_7 = pd.to_numeric(frame["demand_lag_7"], errors="coerce")
    lag_1 = pd.to_numeric(frame["demand_lag_1"], errors="coerce")
    return lag_7.fillna(lag_1).fillna(0.0).to_numpy(dtype=float)


def score(actual, predicted):
    truth = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    total = np.abs(truth).sum()
    return {
        "mae": float(mean_absolute_error(truth, predicted)),
        "rmse": float(np.sqrt(mean_squared_error(truth, predicted))),
        "wape": float(np.abs(truth - predicted).sum() / total) if total else None,
    }


selected_model = make_pipeline(candidate_factories[selected_model_name]())
fit_rows = pd.concat([train, validation], ignore_index=True)
X_fit = fit_rows[FEATURES].copy()
X_test = test[FEATURES].copy()
X_fit["hotel"] = X_fit["hotel"].astype(str)
X_test["hotel"] = X_test["hotel"].astype(str)
selected_model.fit(X_fit, fit_rows["arrivals"].astype(float))

test_prediction = np.maximum(selected_model.predict(X_test), 0.0)
baseline_prediction = seasonal_naive(test)
test_metrics = score(test["arrivals"], test_prediction)
baseline_metrics = score(test["arrivals"], baseline_prediction)

segment_metrics = {}
for hotel, segment in test.groupby("hotel", observed=True):
    segment_X = segment[FEATURES].copy()
    segment_X["hotel"] = segment_X["hotel"].astype(str)
    segment_metrics[str(hotel)] = score(
        segment["arrivals"], np.maximum(selected_model.predict(segment_X), 0.0)
    )

print("Selected model:", test_metrics)
print("Seasonal naive:", baseline_metrics)
print("Test metrics by hotel:")
display(pd.DataFrame(segment_metrics).T)
if test_metrics["mae"] >= baseline_metrics["mae"]:
    print("Warning: selected model did not beat seasonal naive MAE on the test period.")

## 6. Save the artifact for application inference

The application loads the fitted scikit-learn pipeline and metadata from artifacts/models. It never fits a model at API/dashboard startup. The processed daily aggregation is saved separately so the dashboard can forecast the next day locally.

In [ ]:
split_dates = {
    "train_start": str(pd.Timestamp(train_dates[0]).date()),
    "train_end": str(pd.Timestamp(train_dates[-1]).date()),
    "validation_start": str(pd.Timestamp(validation_dates[0]).date()),
    "validation_end": str(pd.Timestamp(validation_dates[-1]).date()),
    "test_start": str(pd.Timestamp(test_dates[0]).date()),
    "test_end": str(pd.Timestamp(test_dates[-1]).date()),
}
model_path = ARTIFACT_DIR / "hotel_arrivals_model.joblib"
metadata_path = ARTIFACT_DIR / "hotel_arrivals_model.json"
daily_path = ARTIFACT_DIR / "hotel_daily_arrivals.csv"

joblib.dump(selected_model, model_path)
daily.to_csv(daily_path, index=False)
metadata = {
    "artifact_format_version": 1,
    "model_version": datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ"),
    "selected_model": selected_model_name,
    "target": "completed reservation arrivals per hotel per calendar day",
    "target_unit": "reservation arrivals (not room nights)",
    "feature_schema": FEATURES,
    "split_dates": split_dates,
    "row_counts": {
        "train": len(train),
        "validation": len(validation),
        "test": len(test),
        **data_audit,
    },
    "validation_metrics": validation_results,
    "test_metrics": {"model": test_metrics, "seasonal_naive": baseline_metrics},
    "test_metrics_by_hotel": segment_metrics,
    "model_beats_baseline_test_mae": test_metrics["mae"] < baseline_metrics["mae"],
    "dataset_file": DATA_PATH.name,
    "dataset_sha256": hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    "limitations": [
        "Source has no room capacity/inventory snapshots or forecast-time prices.",
        "Target counts reservation records, not rooms, room nights, revenue, or price response.",
        "Test is a rolling one-day-ahead holdout that uses prior observed daily counts.",
    ],
}
metadata_path.write_text(json.dumps(metadata, indent=2, allow_nan=False) + "\n", encoding="utf-8")
print(f"Saved model: {model_path.resolve()}")
print(f"Saved metadata: {metadata_path.resolve()}")
print(f"Saved processed daily data: {daily_path.resolve()}")

## 7. Artifact handoff

FastAPI loads hotel_arrivals_model.joblib from MODEL_ARTIFACT_DIR and accepts 28 consecutive daily observations for a one-day forecast at POST /forecast/hotel-arrivals. Streamlit reads the same artifact and the processed daily CSV. The raw source CSV and all generated artifacts remain local and are excluded from Git.